In [3]:
from transformers import AutoTokenizer, AutoConfig, Trainer, TrainingArguments, AutoModelForCausalLM, BitsAndBytesConfig
from llm2vec.models import MistralBiForMNTP
from peft import PeftModel
import torch
from transformers import DataCollatorForLanguageModeling
import pandas as pd
from datasets import Dataset
from torch.utils.data import DataLoader
import os
from peft import LoraConfig, get_peft_model, TaskType

/u/modelfactory/.conda/envs/lab/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [5]:
os.environ["WANDB_DISABLED"] = "true"

In [6]:
sel_cols = ['assetlongdescription_entity_llms', 'failurelocation_original', 
            'assetlongdescription_original', 'mode']
df = pd.read_csv('../processed/asset2item.csv')[sel_cols]
df.rename({'assetlongdescription_original': 'answers.text'}, axis=1, inplace=True)
df['answers.text'] = pd.Series(df['answers.text'], dtype="string")
df.rename({
    'answers.text': 'text'
}, axis=1, inplace=True)
df['text'] = df['text'] + '\n Failure locations:' + df['failurelocation_original']
df_train = df[df['mode']=='train'][['text']]
df_val = df[df['mode']=='val'][['text']]
df_test = df[df['mode']=='test'][['text']]

In [7]:
class CustomDataset(Dataset):
    def __init__(self, df):
        self.df = df
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        if type(idx) == int:
            input = self.df.iloc[i].text
        else:
            input = [self.df.iloc[i].text for i in idx]
        tokenized = tokenizer(input, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        return tokenized

In [8]:
ds_train = CustomDataset(df_train)
ds_val = CustomDataset(df_val)
ds_test = CustomDataset(df_test)

In [9]:
chkpt = "mistralai/Mixtral-8x7B-v0.1"
adapter_chkpt = "mntp-bimixtral-model/checkpoint-500"
use_lora = True

In [10]:
tokenizer = AutoTokenizer.from_pretrained(chkpt)
config = AutoConfig.from_pretrained(chkpt, trust_remote_code=True)
quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(chkpt, quantization_config=quantization_config, device_map="auto")

Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 19/19 [05:40<00:00, 17.91s/it]


In [11]:
if use_lora:
    model.load_adapter(adapter_chkpt)

In [12]:
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [13]:
dataloader = DataLoader(ds_train, collate_fn=data_collator, batch_size=2)

In [14]:
item = next(iter(dataloader)).to(device)
res = model(**item)
tokenizer.batch_decode(torch.argmax(res.logits, axis=2))

["(\riNdEx(\riNdExiNdExiNdExiNdExiNdExiNdExiNdExiNdExiNdExiNdExiNdExiNdExiNdExiNdExiNdExiNdExiNdExiNdExiNdExiNdExiNdExiNdExiNdEx(\riNdExiNdExiNdEx(\r(\r(\r(\r(\r(\r(\r(\r(\r(\r(\r(\r(\r(\r(\r(\r(\r(\r(\r(\r(\r(\r(\rzerwzerwzerwzerw(\rября(\r(\r(\r(\r(\r(\r(\rzerw(\r(\r(\r(\r়ябряября(\r(\rября(\r(\r(\rzerw(\r(\r়়(\r(\riNdEx(\r(\r(\r(\r(\r(\r(\r(\r(\r(\rября(\r(\r(\r(\rября(\r(\r়ябряябряябряябряябряября(\rябряябряябряябряябряябряября(\rябряября(\rябряябряябряябряябряябряябряябряiNdExiNdExябряябряябряябряябряября়়iNdEx(\r়(\r়(\r়iNdExiNdEx়(\riNdExiNdEx়iNdExiNdExiNdEx(\rябряiNdEx়়(\r(\r়(\riNdExiNdExiNdExiNdEx(\r় #  isumulator - Hneumatic - Hadder Type, is categorized as Fixed Asset and has the following boundary: The Pneumatic Accumulator - Bladder Type is a context is comprised of the  - Accank - Bladder - Val in - Valve - if present - Air Line- Valve, Failure locations:{'Gas Pre Valve', 'Gadder', 'T Line Check Valve', if present',\n",
 "#  is T E Ejector - is categorized as Fix

In [15]:
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

In [21]:
training_args = TrainingArguments(
    output_dir="mntp-bimixtral-model",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=False,
    report_to='tensorboard',
    remove_unused_columns=False,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

trainer.train()

Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Epoch,Training Loss,Validation Loss



KeyboardInterrupt



In [20]:
model.save_pretrained("mntp-bimixtral-model/final_model")

NameError: name 'mntp' is not defined